This notebook compares my manually constructed MPOs with the MPOs that Quimb generates using its `qtn.MatrixProductOperator.from_dense()` function.

The motivation for using manually constructed MPOs in the first place was to avoid the need for constructing a dense Laplacian or time-stepping matrix, which would cause my laptop to crash at `n=15` when using LPE2. 

However, as the code below shows, my manual construction does not seem to be as optimal as the Quimb version, leading to bond dimension exploding within the first few steps, and to my laptop crashing generally around 12-13 steps.

In [52]:
import importlib
import utils
importlib.reload(utils)

import numpy as np
import quimb.tensor as qtn

In [61]:



def qtt_diffusion_cores_dd(d: int, cfl: float):
    """
    Direct QTT/MPO cores for the explicit Euler diffusion timestep operator

        A = (1 - 2*cfl) I + cfl S_+ + cfl S_-

    on a size N = 2**d grid.

    Returns cores in shape (left_bond, phys_out, phys_in, right_bond).
    """
    if d < 2:
        raise ValueError("Need d >= 2")

    I = np.array([[1.0, 0.0],
                  [0.0, 1.0]])
    J = np.array([[0.0, 1.0],
                  [0.0, 0.0]])
    JT = J.T

    a = 1.0 - 2.0 * cfl
    b = cfl

    # first core: [ I   JT   J ]
    W_first = np.zeros((1, 2, 2, 3))
    W_first[0, :, :, 0] = I
    W_first[0, :, :, 1] = JT
    W_first[0, :, :, 2] = J

    # middle core:
    # [ I   JT   J ]
    # [ 0    J   0 ]
    # [ 0    0  JT ]
    W_mid = np.zeros((3, 2, 2, 3))
    W_mid[0, :, :, 0] = I
    W_mid[0, :, :, 1] = JT
    W_mid[0, :, :, 2] = J
    W_mid[1, :, :, 1] = J
    W_mid[2, :, :, 2] = JT

    # last core:
    # [ a I ]
    # [ b I ]
    # [ b I ]
    W_last = np.zeros((3, 2, 2, 1))
    W_last[0, :, :, 0] = 2 * I - J - JT
    W_last[1, :, :, 0] = -J
    W_last[2, :, :, 0] = -JT

    cores = [W_first] + [W_mid.copy() for _ in range(d - 2)] + [W_last]
    return cores

def qtt_identity_mpo(n):

    W = np.zeros((1, 1, 2, 2))
    W[0, 0] = np.eye(2)
    arrays = [W.copy() for _ in range(n)] 
    return qtn.MatrixProductOperator(arrays, shape='lrud') # l=left, r=right, u=upper/output, d=lower/input

    

def qtt_diffusion_mpo(d: int, cfl: float):
    cores = qtt_diffusion_cores_dd(d, cfl)

    # convert (left, phys_out, phys_in, right)
    # to      (left, right, phys_out, phys_in)
    cores_q = [np.transpose(W, (0, 3, 1, 2)) for W in cores]

    return qtn.MatrixProductOperator(cores_q)

In [62]:
def shift_plus_cores(n: int):
    if n < 2:
        raise ValueError("n must be > 1")
    
    I = np.array([[1.0, 0.0],
                  [0.0, 1.0]])
    J = np.array([[0.0, 1.0],
                  [0.0, 0.0]])
    JT = J.T

    # first core is [I, J]
    first_core = np.zeros((1,2,2,2))
    first_core[0,:,:,0] = I
    first_core[0,:,:,1] = J
    
    # middle cores are [I  J ]
    #                  [0  JT]
    middle_core = np.zeros((2,2,2,2))
    middle_core[0,:,:,0] = I
    middle_core[0,:,:,1] = J
    middle_core[1,:,:,1] = JT

    # last core is [J ]
    #              [JT]
    last_core = np.zeros((2,2,2,1))
    last_core[0,:,:,0] = J
    last_core[1,:,:,0] = JT

    cores = [first_core] + [middle_core.copy() for _ in range(n-2)] + [last_core]

    cores_quimb = [np.transpose(core, (0,3,1,2)) for core in cores]
    # the paper uses (left bond index, output, input, right bond index)
    # but quimb uses (left bond index, right bond index, output, input)

    return cores_quimb


def shift_plus_mpo(n: int):
    cores_quimb = shift_plus_cores(n)
    return qtn.MatrixProductOperator(cores_quimb, shape='lrud')

def shift_minus_mpo(n: int):
    cores_quimb = shift_plus_cores(n)
    cores_quimb_transposed = [np.transpose(core, (0,1,3,2)).copy() for core in cores_quimb]
    return qtn.MatrixProductOperator(cores_quimb_transposed, shape='lrud')

In [63]:
n=3

Sp = shift_plus_mpo(n)
Sp_dense = np.asarray(Sp.to_dense()).reshape(2**n, 2**n)
print(Sp_dense)
print()

Sm = shift_minus_mpo(n)
Sm_dense = np.asarray(Sm.to_dense()).reshape(2**n, 2**n)
print(Sm_dense)

[[0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]

[[0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]]


In [64]:
def identity_mpo(n):

    W = np.zeros((1, 1, 2, 2))
    W[0, 0] = np.eye(2)
    arrays = [W.copy() for _ in range(n)] 
    return qtn.MatrixProductOperator(arrays, shape='lrud') # l=left, r=right, u=upper/output, d=lower/input

    

def time_step_mpo(n: int, cfl: float, cutoff=1e-12, max_bond=64):

    I  = identity_mpo(n)
    Sp = shift_plus_mpo(n)
    Sm = shift_minus_mpo(n)

    A = (1 - 2 * cfl) * I + cfl * Sp + cfl * Sm
    A.compress(cutoff=cutoff, max_bond=max_bond)
    return A

In [65]:
n = 9
steps = 10               # number of steps required for time evolution

def u(x):
    return np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme

In [66]:
N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)

dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
u0    = u(x)

In [67]:
n = 5
N = 2**n
cfl = 0.1

A_manual = time_step_mpo(n, cfl)
A_manual_dense = np.asarray(A_manual.to_dense()).reshape(N, N)

print(A_manual_dense)
print(A_manual.bond_sizes())

[[0.8 0.1 0.  ... 0.  0.  0. ]
 [0.1 0.8 0.1 ... 0.  0.  0. ]
 [0.  0.1 0.8 ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 0.8 0.1 0. ]
 [0.  0.  0.  ... 0.1 0.8 0.1]
 [0.  0.  0.  ... 0.  0.1 0.8]]
[5, 5, 5, 5, 3]


In [69]:
L = utils.laplacian(N, dx, "dirichlet", "dense")
A_dense = utils.time_step(L, 1, dt, nu)

A_svd = utils.mat_to_qtt_mpo(A_dense, n)
A_manual = time_step_mpo(n, cfl)

A_svd_densified = np.asarray(A_svd.to_dense()).reshape(N,N)
A_manual_densified = np.asarray(A_manual.to_dense()).reshape(N, N)

print("||A_svd - A_dense|| =", np.linalg.norm(A_svd_densified - A_dense))
print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_dense - A_dense))
print("||A_manual - A_svd||  =", np.linalg.norm(A_manual_dense - A_svd_densified))

||A_svd - A_dense|| = 4.27863164248793e-15
||A_manual - A_dense|| = 1.6014269617894046e-15
||A_manual - A_svd||  = 4.4258996490073184e-15


In [41]:
A_manual = qtt_diffusion_mpo(n, cfl)
A_manual_dense = np.asarray(A_manual.to_dense()).reshape(N, N)

print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_dense - A_dense))

||A_manual - A_dense|| = 4.518849411078


In [5]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, [A_manual,], steps)

step  0: 0.020961 s, max bond = 13
step  1: 0.016663 s, max bond = 41
step  2: 0.174073 s, max bond = 64
step  3: 0.900665 s, max bond = 64
step  4: 0.352545 s, max bond = 64
step  5: 0.525948 s, max bond = 64
step  6: 0.702593 s, max bond = 64
step  7: 3.573724 s, max bond = 64
step  8: 3.967774 s, max bond = 64
step  9: 10.763061 s, max bond = 64


In [6]:
mps0 = utils.vec_to_qtt_mps(u0, n)
saved, bonds = utils.evolve_mps_timed(mps0, [A_svd,], steps)

step  0: 0.005343 s, max bond = 5
step  1: 0.004994 s, max bond = 5
step  2: 0.005221 s, max bond = 5
step  3: 0.001687 s, max bond = 5
step  4: 0.002154 s, max bond = 5
step  5: 0.001551 s, max bond = 5
step  6: 0.001440 s, max bond = 5
step  7: 0.002688 s, max bond = 5
step  8: 0.002991 s, max bond = 5
step  9: 0.001476 s, max bond = 5


Below I take a look at the tensor shape and realise that they are quite different.

In [7]:
print("A_manual bond sizes:", A_manual.bond_sizes())
print("A_svd bond sizes:", A_svd.bond_sizes())


for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

for i, T in enumerate(A_svd):
    print(f"A_svd tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 5, 5, 5, 5, 3]
A_svd bond sizes: [3, 3, 3, 3, 3, 3, 3, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 5, 2, 2)
A_manual tensor 5: shape = (5, 5, 2, 2)
A_manual tensor 6: shape = (5, 5, 2, 2)
A_manual tensor 7: shape = (5, 5, 2, 2)
A_manual tensor 8: shape = (5, 3, 2, 2)
A_svd tensor 0: shape = (2, 2, 3)
A_svd tensor 1: shape = (3, 2, 2, 3)
A_svd tensor 2: shape = (3, 2, 2, 3)
A_svd tensor 3: shape = (3, 2, 2, 3)
A_svd tensor 4: shape = (3, 2, 2, 3)
A_svd tensor 5: shape = (3, 2, 2, 3)
A_svd tensor 6: shape = (3, 2, 2, 3)
A_svd tensor 7: shape = (3, 2, 2, 3)
A_svd tensor 8: shape = (3, 2, 2)


Even after compression, the manual MPO remains the same.

In [8]:
A_manual.compress(cutoff=1e-12)
print("A_manual bond sizes:", A_manual.bond_sizes())
for i, T in enumerate(A_manual):
    print(f"A_manual tensor {i}: shape = {T.shape}")

A_manual bond sizes: [5, 5, 5, 5, 5, 5, 5, 5, 3]
A_manual tensor 0: shape = (3, 5, 2, 2)
A_manual tensor 1: shape = (5, 5, 2, 2)
A_manual tensor 2: shape = (5, 5, 2, 2)
A_manual tensor 3: shape = (5, 5, 2, 2)
A_manual tensor 4: shape = (5, 5, 2, 2)
A_manual tensor 5: shape = (5, 5, 2, 2)
A_manual tensor 6: shape = (5, 5, 2, 2)
A_manual tensor 7: shape = (5, 5, 2, 2)
A_manual tensor 8: shape = (5, 3, 2, 2)


Conclusion: `qtn.MatrixProductOperator.from_dense()` optimises the MPO in such a way that my manual construction is not able to emulate, which probably results in the exploding bond dimensions upon evolution.